In [10]:
# Set project paths.
from pathlib import Path
import os
import sys

def find_project_root():
    current = Path.cwd()

    for folder in [current] + list(current.parents):
        if (folder / "Data").exists() and (folder / "Notebooks").exists():
            return folder

    raise FileNotFoundError("Could not find project root. Make sure Data and Notebooks folders exist.")

project_folder = find_project_root()
notebook_folder = project_folder / "Notebooks"

os.chdir(project_folder)

print("Project folder:", project_folder)
print("Notebook folder:", notebook_folder)

Project folder: /Users/mac/Dissertation/SEND-rebuild
Notebook folder: /Users/mac/Dissertation/SEND-rebuild/Notebooks


In [11]:
# Import packages.
import pandas as pd
import numpy as np
import os



In [12]:
# Check raw DEOP folder.
DEOP_Path = "Data/DEOP/Unformatted"

if not os.path.exists(DEOP_Path):
    print("DEOP folder not found")
else:
    print("DEOP folder found")
    print(os.listdir(DEOP_Path)[:5])

DEOP folder found
['DEOP-export-AD Campus Renewables 202302.csv', 'DEOP-export-AD Campus Renewables 202303.csv', 'DEOP-export-AD Campus Renewables 202301.csv', 'DEOP-export-AD Campus Renewables 202304.csv', 'DEOP-export-AD Campus Renewables 202310.csv']


In [13]:
# Read monthly DEOP files.
year = "2022"
start_month = 3
end_month = 12

months = []

for ii in range(start_month, end_month + 1):
    month = year + str("%02d" % ii)
    print(month)

    df = pd.read_csv(
        f"{DEOP_Path}/DEOP-export-AD Campus Renewables {month}.csv",
        sep=";",
        dtype={
            "date0": str,
            "time0": str,
            "power-con-ave": float,
            "power-gen-wt-ave": float,
            "power-gen-pv-ave": float,
        },
    )

    format_date = "%m/%d/%Y %I:%M:%S %p"

    df["DateTime"] = pd.to_datetime(
        df["date0"] + " " + df["time0"],
        format=format_date,
    )

    months.append(df)

combined_months = pd.concat(months, ignore_index=True)

combined_months.head()

202203
202204
202205
202206
202207
202208
202209
202210
202211
202212


,date0,time0,power-ss13-max,power-ss13-min,power-ss13-ave,date1,time1,power-gen-max,power-gen-min,power-gen-ave,...,time4,power-gen-pv-max,power-gen-pv-min,power-gen-pv-ave,date5,time5,storage-charge-max,storage-charge-min,storage-charge-ave,DateTime
0,03/01/2022,12:00:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12:00:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-03-01 00:00:00
1,03/01/2022,12:05:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12:05:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-03-01 00:05:00
2,03/01/2022,12:10:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12:10:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-03-01 00:10:00
3,03/01/2022,12:15:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12:15:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-03-01 00:15:00
4,03/01/2022,12:20:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12:20:00 am,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-03-01 00:20:00


In [14]:
# Filter selected year.
combined_months.index = combined_months["DateTime"]

combined_months = combined_months.loc[
    (combined_months.index >= f"{year}-01-01 00:05")
    & (combined_months.index <= f"{year}-12-31 23:55")
]

In [15]:
# Clean invalid values.
combined_months = combined_months.fillna(0)

combined_months["power-con-ave"] = combined_months["power-con-ave"].clip(lower=0)
combined_months["power-gen-wt-ave"] = combined_months["power-gen-wt-ave"].clip(lower=0)
combined_months["power-gen-pv-ave"] = combined_months["power-gen-pv-ave"].clip(lower=0)

In [16]:
# Fix timestamps.
combined_months = combined_months.tz_localize(
    "Europe/London",
    ambiguous="NaT",
    nonexistent="NaT",
)

combined_months = combined_months.tz_convert("UTC").tz_localize(None)

combined_months = combined_months.loc[~combined_months.index.isnull()]

full_index = pd.date_range(
    start=combined_months.index.min(),
    end=combined_months.index.max(),
    freq="5min",
)

combined_months = combined_months.reindex(full_index).fillna(0)

combined_months["DateTime"] = combined_months.index

In [17]:
# Interpolate consumption.
averaged_comb_months = combined_months.copy()

averaged_comb_months["power-con-ave"] = averaged_comb_months["power-con-ave"].replace(0, np.nan)
averaged_comb_months["power-con-ave"] = averaged_comb_months["power-con-ave"].interpolate()

averaged_comb_months["power-con-ave"] = averaged_comb_months["power-con-ave"].fillna(0)

In [18]:
# Save DEOP outputs.
combined_months.to_csv(
    f"Data/DEOP/{year}_DEOP.csv",
    index=False,
    columns=["power-con-ave", "power-gen-wt-ave", "power-gen-pv-ave", "DateTime"],
)

averaged_comb_months.to_csv(
    f"Data/DEOP/{year}_DEOP_Interp.csv",
    index=False,
    columns=["power-con-ave", "power-gen-wt-ave", "power-gen-pv-ave", "DateTime"],
)